In [ ]:
import os, sys, json, glob, subprocess
from pathlib import Path
import torch

MODEL = 'gru'
BLEU_SAMPLES = 1000     # val pairs to greedy-decode for BLEU/METEOR (0 = skip)

DATASET_PATH = Path(glob.glob('/kaggle/input/**/train.tsv', recursive=True)[0]).parent
TOKENIZER_MODEL = Path(glob.glob('/kaggle/input/**/spm_en_id.model', recursive=True)[0])
USE_METEOR = True       # Indonesian-aware METEOR (needs nltk data + Sastrawi below)

REPO_URL = 'https://github.com/0wLzz/Edge-NMT.git'
if not os.path.isdir('Edge-NMT'):
    subprocess.run(['git','clone','--depth','1',REPO_URL], check=True)
    
os.chdir('/kaggle/working/Edge-NMT')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
print('dataset:', DATASET_PATH)

In [ ]:
# Dependencies (same core set as the search; + nltk/Sastrawi for METEOR)
pkgs = ['sentencepiece','sacrebleu','pyyaml','optuna','torchinfo', 'coremltools']
if USE_METEOR:
    pkgs += ['nltk','Sastrawi']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=True)

if USE_METEOR:
    import nltk
    nltk.download('wordnet'); nltk.download('omw-1.4')

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), 'sm_%d%d' % cap)
else:
    print('No GPU detected, using CPU.')

In [ ]:
CFG = 'configs/config.yaml'

def run_module(mod, *args):
    print('RUN', mod, *args, flush=True)
    subprocess.run([sys.executable,'-m',mod,*map(str,args)], check=True)

exp_args = [
    '--arch', MODEL,
    '--config', CFG,
    '--baseline',
    '--prune-train',
    '--prune-post',

    '--dataset-dir', DATASET_PATH,
    '--tokenizer-model', TOKENIZER_MODEL,
    '--bleu-samples', BLEU_SAMPLES,
    '--epochs', 15,
]

if USE_METEOR:
    exp_args.append('--meteor')

# Run Experiments
run_module('model.experiments.prune_post_vs_train', *exp_args)